# Assignment 1
**Credits**: Federico Ruggeri, Giulia Grundler, Paolo Torroni

**Keywords**: Unfair Clause Detection, Multi-label Classification, RNNs, Transformers, Huggingface

# Contact
For any doubt, question, issue or help, you can always contact us at the following email addresses:

Teaching Assistants:

- Federico Ruggeri -> federico.ruggeri6@unibo.it
- Giulia Grundler -> giulia.grundler2@unibo.it

Professor:
- Paolo Torroni -> p.torroni@unibo.it

# Introduction

You are asked to address **unfair clause detection** in online Terms of Service (ToS), following the
[CLAUDETTE](http://claudette.eui.eu/) line of work.

Consumer contracts are long, nobody reads them, and some of their clauses are *potentially unfair*: they grant the
service provider rights that a consumer-protection authority would likely consider abusive. The task is to read one
clause at a time and flag which kinds of unfairness it carries.

## Problem Definition

Given the text of a single contract clause, predict which of the following five unfairness categories apply:

| Label | Category | Meaning |
|-------|----------|---------|
| `A`   | Arbitration | Disputes must go to arbitration instead of a court. |
| `CH`  | Unilateral change | The provider may change the contract or the service unilaterally. |
| `CR`  | Content removal | The provider may remove or block user content at its own discretion. |
| `LTD` | Limitation of liability | The provider limits or excludes its liability towards the consumer. |
| `TER` | Unilateral termination | The provider may terminate the contract or the user account at its own discretion. |

This is a **multi-label** problem, **not** multi-class:

* a clause may carry **none** of the five categories (the large majority of clauses);
* a clause may carry **more than one** category at the same time.

Each category is therefore an independent binary decision, and your model has **five** outputs.

### Examples

#### A - Arbitration

''*the arbitrator has exclusive authority to resolve any dispute relating to the interpretation , applicability , or
enforceability of this binding arbitration agreement .*''

#### CH - Unilateral change

''*we reserve the right to modify any provision hereof from time to time , in our sole discretion , and such
modification shall be effective immediately upon its posting on the website .*''

#### CR - Content removal

''*we will not edit or monitor user-provided content , but we reserve the right to remove any user provided content
which comes to our attention and which , in our sole discretion , breaches this agreement .*''

#### LTD - Limitation of liability

''*the collective liability of mozilla and the indemnified parties under this agreement will not exceed $ 500 -lrb-
five hundred dollars -rrb- .*''

#### TER - Unilateral termination

''*in case of a dispute with the member who owns the site , we are allowed to ban this member and remove him/her from
the service at our discretion .*''

#### Multiple labels on the same clause

''*we may remove your dna results and/or dna reports and/or terminate your membership at any time , without notice .*''
$\rightarrow$ `CR` **and** `TER`.

''*we reserve the right to modify or terminate the service or your access to the service for any reason , without
notice , at any time , and without liability to you .*'' $\rightarrow$ `CH` **and** `TER`.

#### No label (fair clause)

''*this dispute resolution provision will be governed by the federal arbitration act .*''

Note how close this last clause is to the `A` example: the topic is arbitration, but the clause does not impose it.
**Topic is not the label.**

# [Task 1 - 1.0 points] Corpus

We use **ToS-100**: 100 Terms of Service documents split into 20,417 clauses, each annotated with the five
unfairness categories above.

The dataset is available on Zenodo: **[ToS-100](https://doi.org/10.5281/zenodo.22691656)**

The archive contains three `.csv` files: `train.csv`, `validation.csv` and `test.csv`.

### Dataset Description

* Clauses come from real Terms of Service of online platforms (Mozilla, Netflix, Spotify, ...).
* The text is already **lowercased** and **tokenized** in Penn Treebank style: brackets appear as `-lrb-` / `-rrb-`,
  quotes as `` `` `` and `''`, and clitics are detached (`mozilla 's`, `do n't`).
* Splits are **by document**, not by clause: all clauses of one contract stay in the same split. Contracts copy each
  other almost verbatim, so a clause-level split would leak the test set into training.

| Split | Documents | Clauses | A | CH | CR | LTD | TER |
|-------|-----------|---------|---|----|----|-----|-----|
| train | 80 | 15,837 | 75 | 268 | 165 | 504 | 318 |
| validation | 10 | 2,548 | 20 | 43 | 32 | 75 | 59 |
| test | 10 | 2,032 | 11 | 33 | 19 | 47 | 43 |

The corpus is **heavily imbalanced**: 18,843 clauses out of 20,417 (92.3%) carry no label at all.

### Example

| column | value |
|--------|-------|
| `document_ID` | `1` |
| `document` | `MyHeritage` |
| `text` | `we may remove your dna results and/or dna reports and/or terminate your membership at any time , without notice .` |
| `A` | `0` |
| `CH` | `0` |
| `CR` | `1` |
| `LTD` | `0` |
| `TER` | `1` |

The `*_targets` columns in the original release point to a knowledge base of legal rationales. **They are not used in
this assignment** and can be dropped.

### Instructions

1. **Download** the dataset from Zenodo.
2. **Load** the three CSV files and encode them as `pandas.DataFrame`.
3. **Remove unwanted columns**: keep only `document_ID`, `text`, and the five label columns `A`, `CH`, `CR`, `LTD`,
   `TER`.
4. **Build the target**: for each clause, a 5-dimensional binary vector `[A, CH, CR, LTD, TER]`.
5. **Report** basic statistics of each split: number of clauses, number of positives per category, number of clauses
   with 0 / 1 / 2+ labels.

**Note**: do **not** re-split the data. The provided splits are document-based on purpose.

# [Task 2 - 0.5 points] Data Cleaning and Label Analysis

### [Task 2a - 0.25 points] Data Cleaning

The text is legal prose that has already gone through a tokenizer, which leaves artifacts behind.

#### Instructions

* **Restore bracket tokens**: `-lrb-` $\rightarrow$ `(`, `-rrb-` $\rightarrow$ `)`, and likewise `-lsb-` / `-rsb-`.
* **Normalize quote tokens** (`` `` ``, `''`, `` ` ``, `'`).
* **Reattach clitics** or remove them (`mozilla 's` $\rightarrow$ `mozilla's`, `do n't` $\rightarrow$ `don't`).
* **Remove clause numbering and bullets** (e.g. `1.`, `2.3.1`, `a )`, `-`) at the beginning of a clause.
* **Normalize whitespace** and remove leftover special characters.
* **Perform lemmatization** to reduce words to their base form.

**Note**: be careful with what you remove. Modal verbs (`may`, `shall`, `will`), negations and pronouns (`we`, `you`)
carry most of the signal in this task, so an aggressive stopword list can hurt you. Justify your choices.

### [Task 2b - 0.25 points] Label Analysis

#### Instructions

* Compute the **positive rate** of each of the five categories on the training set.
* Compute and comment the **co-occurrence** between categories (e.g. a 5x5 matrix of joint counts).
* Given the imbalance, **choose and justify** a mitigation strategy for training, e.g.:
  * per-class positive weighting in the loss (`pos_weight` in `BCEWithLogitsLoss`);
  * resampling of positive clauses;
  * no mitigation, with threshold tuning at inference time instead.
* You will apply the chosen strategy in Task 5.

# [Task 3 - 0.5 points] Text Encoding

To train a neural unfair-clause classifier, you first need to encode text into numerical format.

### Instructions

* Embed words using **GloVe embeddings**.
* You are **free** to pick any embedding dimension.

### What about OOV tokens?

* All the tokens in the **training** set that are not in GloVe **must** be added to the vocabulary.
* For the remaining tokens (i.e., OOV in the validation and test sets), you have to assign them a **special token**
  (e.g., `<UNK>`) and a **static** embedding.
* You are **free** to define the static embedding using any strategy (e.g., random, neighbourhood, etc...)

**Note**: legal text has its own vocabulary (`hereof`, `indemnify`, `arbitrator`), so expect a non-trivial OOV rate.
Report it.

### More about OOV

For a given token:

* **If in train set**: add to vocabulary and assign an embedding (use GloVe if token in GloVe, custom embedding
  otherwise).
* **If in val/test set**: assign special token if not in vocabulary and assign custom embedding.

Your vocabulary **should**:

* Contain all tokens in train set; or
* Union of tokens in train set and in GloVe $\rightarrow$ we make use of existing knowledge!

# [Task 4 - 1.0 points] Model definition

You are now tasked to define your unfair clause classifier.

### Instructions

* **Baseline**: implement a Bidirectional LSTM with a Dense layer on top.

* **Stacked**: add an additional Bidirectional LSTM layer to the Baseline model.

**Note**: You are **free** to experiment with hyper-parameters.

### Multi-label output

Since a clause can carry several categories at once, the classification head is **not** a softmax over 5 classes.

* The output layer has **5 units**, one per category.
* Use a **sigmoid** activation (or raw logits with `BCEWithLogitsLoss` / `from_logits=True`).
* Use **binary cross-entropy** as the loss, **not** categorical cross-entropy.
* At inference time, a category is predicted when its probability exceeds a **threshold** (0.5 by default). You are
  **free** to tune the threshold on the **validation** set only.

### Token to embedding mapping

You can follow two approaches for encoding tokens in your classifier.

### Work directly with embeddings

- Compute the embedding of each input token
- Feed the mini-batches of shape ``(batch_size, # tokens, embedding_dim)`` to your model

### Work with Embedding layer

- Encode input tokens to token ids
- Define a Embedding layer as the first layer of your model
- Compute the embedding matrix of all known tokens (i.e., tokens in your vocabulary)
- Initialize the Embedding layer with the computed embedding matrix
- You are **free** to set the Embedding layer trainable or not

In [ ]:
embedding = tf.keras.layers.Embedding(input_dim=vocab_size,
                                      output_dim=embedding_dimension,
                                      weights=[embedding_matrix],
                                      mask_zero=True,                   # automatically masks padding tokens
                                      name='encoder_embedding')

# [Task 5 - 1.0 points] Training and Evaluation

You are now tasked to train and evaluate the Baseline and Stacked models.

### Instructions

* Pick **at least** three seeds for robust estimation.
* Train **all** models on the train set, applying the imbalance strategy chosen in Task 2b.
* Evaluate **all** models on the validation and test sets.
* Compute **macro** F1-score, precision, and recall over the five categories on the validation set.
* Also report the **per-category** F1-score: the macro average hides the fact that `A` has 75 positive training
  clauses and `LTD` has 504.
* Report average and standard deviation measures over seeds for each metric.
* Pick the **best** performing model according to the observed validation set performance (use macro F1-score).

**Note**: macro F1 here is the average of the five **binary** F1-scores, one per category, each computed on the
positive class. Accuracy is meaningless on this corpus (a model predicting all zeros scores above 92%): **do not**
report it as your main metric.

# [Task 6 - 1.0 points] Transformers

In this section, you will use a transformer model pre-trained on legal text, namely
[LEGAL-BERT](https://huggingface.co/nlpaueb/legal-bert-base-uncased).

### Instructions
- **Load the Tokenizer and Model**

  Configure the classification head for multi-label: with Huggingface, set
  `problem_type="multi_label_classification"` and `num_labels=5` in `AutoModelForSequenceClassification`.

- **Preprocess the Dataset**:
   You will need to preprocess your dataset to prepare it for input into the model. Tokenize your text data using the
   appropriate tokenizer and ensure it is formatted correctly. Labels must be a `float` vector of size 5.

- **Train the Model**:
   Use the `Trainer` to train the model on your training data.

- **Evaluate the Model on the Test Set** using the same metrics used for LSTM-based models.

# [Task 7 - 0.5 points] Error Analysis

After evaluating the model, perform a brief error analysis on the **test set**:

### Instructions

 - Review the results and identify common errors.

 - Summarize your findings regarding the errors and their impact on performance (e.g. but not limited to
   Out-of-Vocabulary (OOV) words, class imbalance, confusion between related categories such as `CR` and `TER`, clauses
   carrying multiple labels, and performance differences between the custom model and the transformer...)

 - Suggest possible solutions to address the identified errors.

# [Task 8 - 0.5 points] Report

Wrap up your experiment in a short report (up to 2 pages).

### Instructions

* Use the NLP course report template.
* Summarize each task in the report following the provided template.

### Recommendations

The report is **not a copy-paste** of graphs, tables, and command outputs.

* Summarize classification performance in Table format.
* **Do not** report command outputs or screenshots.
* Report learning curves in Figure format.
* The error analysis section should summarize your findings.

# Submission

* **Submit** your report in PDF format.
* **Submit** your python notebook.
* Make sure your notebook is **well organized**, with no temporary code, commented sections, tests, etc...
* You can upload **model weights** in a cloud repository and report the link in the report.

## Bonus Points
Bonus points are arbitrarily assigned based on significant contributions such as:
- Outstanding error analysis
- Masterclass code organization
- Suitable extensions

**Note**: bonus points are only assigned if all task points are attributed (i.e., 6/6).

**Possible Suggestions for Bonus Points:**
- **Compare against a general-domain transformer** (e.g. `bert-base-uncased`) to quantify what legal pre-training is
  worth on this task.
- **Try legal word embeddings**: [Law2Vec](https://archive.org/details/Law2Vec) is a word2vec model trained on 123k
  legal documents (UK, EU, Canadian, Australian, US and Japanese legislation), released in the public domain as a
  plain word2vec text file:

  ```python
  # !wget https://archive.org/download/Law2Vec/Law2Vec.100d.txt
  from gensim.models import KeyedVectors
  law2vec = KeyedVectors.load_word2vec_format('Law2Vec.100d.txt', binary=False)
  ```

  Report the OOV rate of both models on your vocabulary and what the swap does to your F1-score. Two things to watch:
  Law2Vec has a smaller vocabulary than GloVe (169k against 400k), and it was trained with every digit replaced by
  `D`, so `500` is OOV unless you normalize numbers the same way.
- **Compare imbalance strategies**: class weighting vs. resampling vs. threshold tuning, with the same architecture.
- **Exploit document structure**: clauses are not independent, they come from a contract with a context. Try feeding
  the neighbouring clauses.
- **Experiment with other custom architectures or models from HuggingFace**.

# FAQ

Please check this frequently asked questions before contacting us

### Multi-label vs. multi-class

A clause can be unfair under several categories at once, and most clauses are unfair under none. Do **not** collapse
the five columns into a single categorical label, and do **not** drop the fair clauses.

### Trainable Embeddings

You are **free** to define a trainable or non-trainable Embedding layer to load the GloVe embeddings.

### Model architecture

You **should not** change the architecture of a model (i.e., its layers).

However, you are **free** to play with their hyper-parameters.

### Neural Libraries

You are **free** to use any library of your choice to implement the networks (e.g., Keras, Tensorflow, PyTorch, JAX,
etc...)

### Robust Evaluation

Each model is trained with at least 3 random seeds.

Task 5 requires you to compute the average performance over the 3 seeds and its corresponding standard deviation.

### Expected Results

The task is hard and the corpus is small in terms of positive examples.

* Recurrent baselines are expected around **40-55** macro F1-score.
* LEGAL-BERT is expected around **60-75** macro F1-score.
* Per-category scores vary a lot: `LTD` and `TER` are the easiest, `A` is the hardest (11 positive test clauses).

These are indicative ranges, not thresholds for grading: a well-analysed lower score is worth more than an unexplained
higher one.

### Threshold tuning

You may tune the decision threshold (globally or per category), but **only** on the validation set. Report the
threshold you used.

### Model Selection for Analysis

To carry out the error analysis you are **free** to either

* Pick examples or perform comparisons with an individual seed run model (e.g., Baseline seed 1337)
* Perform ensembling via, for instance, majority voting to obtain a single model.

### Error Analysis

Some topics for discussion include:
   * Precision/Recall curves.
   * Per-category confusion matrices.
   * Specific misclassified clauses.

### Dataset References

If you want to know more about the corpus and the task:

* Lippi et al., 2019. *CLAUDETTE: an Automated Detector of Potentially Unfair Clauses in Online Terms of Service*.
  Artificial Intelligence and Law.
* Ruggeri et al., 2022. *Detecting and Explaining Unfairness in Consumer Contracts through Memory Networks*.
  Artificial Intelligence and Law.

# The End

Feel free to reach out for questions/doubts!